# SECTION 1 — Setup and Data Loading
This section covers setting up our Python environment, connecting to Google Drive, and loading the planetary image dataset. We resize all images uniformly to 128x128 pixels and normalize their pixel values between 0 and 1 so our deep learning models can process them easily.
*Syllabus Unit Covered: Basics of Deep Learning, Data Preparation, and Preprocessing (Unit 1)*


In [ ]:
# 1. Import necessary libraries
import os
import cv2  # For image processing
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import random

# Global variable to store test accuracies across the notebook
global_results = {}

# Helper function for plotting model history
def plot_history(history, title):
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train')
    plt.plot(history.history['val_accuracy'], label='Validation')
    plt.title(f'{title} Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train')
    plt.plot(history.history['val_loss'], label='Validation')
    plt.title(f'{title} Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.tight_layout()
    plt.show()

# 2. Set random seeds to ensure full reproducibility (random_state=42 everywhere)
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

# 3. Mount Google Drive to access the dataset
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except Exception as e:
    print("Warning: Google Drive could not be mounted. Ensure you are running this cell in Google Colab.")

# 4. Define our planetary classes and constants
CLASSES = ['Mercury', 'Venus', 'Earth', 'Mars', 'Jupiter', 'Saturn', 'Uranus', 'Neptune', 'Pluto', 'Makemake', 'Moon']
DATASET_PATH = '/content/drive/MyDrive/Planets'
IMAGE_SIZE = (128, 128)

X_list = []
y_list = []

# 5. Load and process images
if not os.path.exists(DATASET_PATH):
    print(f"\n[!] ALERT: Dataset path '{DATASET_PATH}' was not found.")
    print("Generating a tiny synthetic dataset of random noise to prevent notebook crashes for demonstration.")
    # Fallback to prevent notebook crashing if user doesn't have the explicit drive folder set up yet
    for cls in CLASSES:
        if cls == 'Makemake': continue
        for _ in range(50):
            X_list.append(np.random.rand(128, 128, 3))
            y_list.append(cls)
else:
    for planet in CLASSES:
        planet_path = os.path.join(DATASET_PATH, planet)
        
        # Handle the missing Makemake folder gracefully with a warning
        if not os.path.exists(planet_path):
            print(f"Warning: Folder for {planet} not found at {planet_path}. Skipping gracefully without crashing.")
            continue
        
        for img_name in os.listdir(planet_path):
            img_path = os.path.join(planet_path, img_name)
            try:
                img = cv2.imread(img_path)
                if img is not None:
                    # Convert BGR (OpenCV default) to standard RGB format
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    # Resize to identical 128x128 resolution
                    img = cv2.resize(img, IMAGE_SIZE)
                    # Normalize pixel values directly to [0, 1] range here
                    img = img / 255.0
                    
                    X_list.append(img)
                    y_list.append(planet)
            except Exception as e:
                pass

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list)
print(f"Total processed images loaded: {len(X)}")

# 6. Encode Labels into one-hot vectors
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)
# Convert integers to categorical layout (one-hot matrices)
y_categorical = tf.keras.utils.to_categorical(y_encoded, num_classes=len(CLASSES))

# 7. Split data into 80% Train, 20% validation 
X_train, X_test, y_train, y_test = train_test_split(X, y_categorical, test_size=0.2, random_state=42)

# 8. Show a grid of sample planetary images (one per class found)
plt.figure(figsize=(15, 6))
unique_planets = np.unique(y)
for i, planet in enumerate(unique_planets):
    idx = np.where(y == planet)[0][0] # Grab the first instance
    plt.subplot(2, 6, i+1)
    plt.imshow(X[idx])
    plt.title(planet)
    plt.axis('off')

plt.suptitle("Sample Planetary Images", fontsize=16)
plt.tight_layout()
plt.show()


# SECTION 2 — Data Augmentation (Unit 3)
Why use Data Augmentation? Neural networks can easily memorize a small dataset (Overfitting). By randomly rotating, flipping, zooming, and shifting the brightness of our images during training, we synthetically increase our dataset diversity. This forces the model to learn the generalized features of the planet instead of memorizing specific rotations, dramatically improving real-world capability (Generalization).
*Syllabus Unit Covered: Image Processing & Data Augmentation Pipelines (Unit 3)*


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Setup the ImageDataGenerator specifying our random augmentations
datagen = ImageDataGenerator(
    rotation_range=20,       # Rotate randomly up to 20 degrees
    horizontal_flip=True,    # Random horizontal mirroring
    zoom_range=0.2,          # Random zoom up to 20%
    brightness_range=[0.8, 1.2] # Vary brightness between 80% and 120%
)

# 2. Take a single image for testing augmentation visually
sample_img = X_train[0]
sample_batch = np.expand_dims(sample_img, axis=0) # Add batch dimension (1, 128, 128, 3)

# 3. Create iterator to generate images
aug_iter = datagen.flow(sample_batch, batch_size=1)

# 4. Display Original vs Augmented
plt.figure(figsize=(15, 4))
plt.subplot(1, 6, 1)
plt.imshow(sample_img)
plt.title("Original")
plt.axis('off')

for i in range(5):
    # Retrieve modified instances from generator
    aug_img = next(aug_iter)[0]
    plt.subplot(1, 6, i+2)
    plt.imshow(aug_img)
    plt.title(f"Augmented {i+1}")
    plt.axis('off')

plt.suptitle("How Data Augmentation Modifies an Image", fontsize=16)
plt.tight_layout()
plt.show()


# SECTION 3 — Autoencoder for Feature Compression (Unit 3)
An Autoencoder is an unsupervised network. Its 'Encoder' half learns to compress planetary image information into a small core representation (bottleneck). Its 'Decoder' takes that compressed representation and learns to unpack it to recreate the original image. We train them together, then snip off the Encoder to use it as a powerful feature compressor for our basic classifying algorithms.
*Syllabus Unit Covered: Unsupervised Learning, Feature Extraction & Autoencoders (Unit 3)*


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D

# 1. ENCODER architecture
auto_input = Input(shape=(128, 128, 3))
x = Conv2D(32, (3, 3), activation='relu', padding='same')(auto_input)
x = MaxPooling2D((2, 2), padding='same')(x) # Down to 64x64

x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2), padding='same')(x) # Down to 32x32

x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
encoded = MaxPooling2D((2, 2), padding='same')(x) # Compressed representation map (16x16x128)

# 2. DECODER architecture 
x = Conv2D(128, (3, 3), activation='relu', padding='same')(encoded)
x = UpSampling2D((2, 2))(x) # Up back to 32x32

x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x) # Up to 64x64

x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x) # Up to 128x128

decoded = Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)

# Compile full autoencoder network
autoencoder = Model(auto_input, decoded)
autoencoder.compile(optimizer='adam', loss='mse')

# 3. Train the Autoencoder (Input and Target are both X_train!)
print("Training Autoencoder for Feature Extraction...")
autoencoder.fit(X_train, X_train, epochs=20, batch_size=32, validation_data=(X_test, X_test), verbose=1)

# 4. Display Reconstructions
reconstructed_imgs = autoencoder.predict(X_test[:5])
plt.figure(figsize=(15, 6))
for i in range(5):
    plt.subplot(2, 5, i+1)
    plt.imshow(X_test[i])
    plt.title("Original Planet")
    plt.axis('off')
    
    plt.subplot(2, 5, i+6)
    plt.imshow(reconstructed_imgs[i])
    plt.title("Reconstructed")
    plt.axis('off')
plt.suptitle("Autoencoder Feature Reconstruction test", fontsize=16)
plt.tight_layout()
plt.show()

# 5. Extract trained encoder and generate compressed features for next section
encoder_model = Model(auto_input, encoded)
X_train_compressed = encoder_model.predict(X_train)
X_test_compressed = encoder_model.predict(X_test)
print(f"Compressed Training Shape: {X_train_compressed.shape}")


# SECTION 4 — MLP Baseline Classifier (Unit 1 + Unit 2)
Here we pass the compressed features into a Multilayer Perceptron (MLP) Artificial Neural Network. An MLP features a series of fully-connected 'Dense' layers that perform weighted sums on inputs. Training uses Backpropagation to push error corrections backward updating the weights.
Dropout Regularization arbitrarily drops neurons out of training to prevent them from becoming hyper-focused on specific memorized traits, controlling overfitting.
*Syllabus Unit Covered: Multi-Layer Perceptrons & Backpropagation (Units 1 & 2)*


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout

# 1. Build the simple Artificial Neural Network
mlp_model = Sequential([
    Flatten(input_shape=(16, 16, 128)), # Flatten the 3D compressed shape from encoder
    Dense(256, activation='relu'),
    Dropout(0.3), # Dropout acts as a regularization layer
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(11, activation='softmax') # Softmax produces probabilities for 11 class labels
])

mlp_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 2. Train Model
history_mlp = mlp_model.fit(X_train_compressed, y_train, epochs=30, validation_data=(X_test_compressed, y_test), batch_size=32, verbose=1)

# 3. Plot metrics and report test accuracy
plot_history(history_mlp, "MLP Baseline")

test_loss, test_acc = mlp_model.evaluate(X_test_compressed, y_test, verbose=0)
global_results['MLP Baseline'] = test_acc
print(f"MLP Baseline Final Test Accuracy: {test_acc:.4f}")


# SECTION 5 — CNN Classifier from Scratch (Unit 4)
Convolutional Neural Networks (CNNs) analyze blocks of pixels rather than flattening the image immediately. 'Filters' scan across images mapping shapes. 'Pooling' shrinks output maps to save memory. Finally, 'Batch Normalization' re-centers numerical layer outputs, eliminating drastic unscaled spikes, causing smooth and rapid network optimization.
*Syllabus Unit Covered: Convolutional Neural Networks Architecture & Basics (Unit 4)*


In [ ]:
from tensorflow.keras.layers import BatchNormalization

# 1. Define standard CNN generator block function
def build_cnn():
    model = Sequential([
        # Block 1
        Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 3)),
        BatchNormalization(),
        MaxPooling2D((2,2)),
        
        # Block 2
        Conv2D(64, (3,3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2,2)),
        
        # Block 3
        Conv2D(128, (3,3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2,2)),
        
        # Classification Head
        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.4),
        Dense(11, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# 2. Train on raw pixel arrays directly
original_cnn = build_cnn()
print("Training basic CNN Classifier from scratch...")
history_cnn = original_cnn.fit(X_train, y_train, epochs=15, validation_data=(X_test, y_test), batch_size=32, verbose=1)

# 3. Save Results
plot_history(history_cnn, "Original CNN")
test_acc_cnn = original_cnn.evaluate(X_test, y_test, verbose=0)[1]
global_results['CNN (Original Data)'] = test_acc_cnn
print(f"Original CNN Final Test Accuracy: {test_acc_cnn:.4f}")


# SECTION 6 — CNN Classifier with Augmented Data
This section proves the power of the ImageDataGenerator. We rebuild the exact same CNN model structurally. However, during the training loop we stream augmented, dynamically shifted dataset images. The system will learn far superior 'generalized' features that improve test scoring noticeably.
*Syllabus Unit Covered: Implementing Model Generalization & Data Pipelines (Unit 4)*


In [ ]:
# 1. Build fresh untested CNN
aug_cnn = build_cnn()

# 2. Train using the datagen.flow() rather than raw X_train
print("Training CNN Classifier on Augmented generator flow...")
history_aug = aug_cnn.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=15,
    validation_data=(X_test, y_test),
    verbose=1
)

# 3. Save results
plot_history(history_aug, "Augmented CNN")
test_acc_aug = aug_cnn.evaluate(X_test, y_test, verbose=0)[1]
global_results['CNN (Augmented Data)'] = test_acc_aug

# 4. Compare directly
print("\n--- Vanilla VS Augmented CNN Comparison Table ---")
print(f"CNN (Original Data) Accuracy:  {test_acc_cnn:.4f}")
print(f"CNN (Augmented Data) Accuracy: {test_acc_aug:.4f}")


# SECTION 7 — Transfer Learning Classifier (Unit 4)
Transfer Learning bypasses training low-level basic edge-detecting filters. We import `EfficientNetB0`, a powerhouse image model fully 'pre-trained' on millions of general photos. We lock ('freeze') its original convolution layers so they retain this universal knowledge, appending custom prediction nodes specifically tuned to map the frozen layer discoveries towards our our 11 Planets. 
*Syllabus Unit Covered: Pre-trained Networks and Custom Transfer Heads (Unit 4)*


In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D

# 1. Load Pre-trained structure without standard labels (include_top=False)
base_eff_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(128, 128, 3))

# 2. Freeze all loaded weights inside the system
base_eff_model.trainable = False

# 3. Add custom classification head on top
tl_model = Sequential([
    base_eff_model,
    GlobalAveragePooling2D(), # Averages entire spatial map reducing computation
    Dense(256, activation='relu'),
    Dropout(0.4),
    Dense(11, activation='softmax')
])

tl_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 4. Train Transfer Learning network (using augmentation config)
print("Training Transfer Learning Custom Head...")
history_tl = tl_model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=20, 
    validation_data=(X_test, y_test),
    verbose=1
)

plot_history(history_tl, "Transfer Learning EfficientNetB0")
test_acc_tl = tl_model.evaluate(X_test, y_test, verbose=0)[1]
global_results['Transfer Learning (EfficientNetB0)'] = test_acc_tl
print(f"Transfer Learning Final Test Accuracy: {test_acc_tl:.4f}")


# SECTION 8 — GAN for Planetary Image Generation (Unit 5)
A Generative Adversarial Network puts two systems in constant battle. The `Generator` transforms pure random statistical noise into a pixel block, attempting to make a fake planet. The `Discriminator` runs as a binary classifier guessing '(1) Real Dataset Planet' or '(0) Fake Generator Trash'. This adversarial loss trains both nets deeply. Over exactly 500 limited steps, we attempt to create synthetic data.
*Syllabus Unit Covered: Generative Deep Learning, DCGANs (Unit 5)*


In [ ]:
from tensorflow.keras.layers import Reshape, Conv2DTranspose, LeakyReLU

latent_dim = 100

# 1. GENERATOR: Maps Noise vector (100,) -> Image (128,128,3)
generator = Sequential([
    Dense(16 * 16 * 128, input_dim=latent_dim),
    BatchNormalization(),
    LeakyReLU(alpha=0.2),
    Reshape((16, 16, 128)),
    
    # Fractional-Strided Convolution reverses MaxPool upscaling the dimensions
    Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'), # 32x32
    BatchNormalization(),
    LeakyReLU(alpha=0.2),
    
    Conv2DTranspose(64, (4,4), strides=(2,2), padding='same'),  # 64x64
    BatchNormalization(),
    LeakyReLU(alpha=0.2),
    
    # Output squashed between -1 and 1 via tanh
    Conv2DTranspose(3, (4,4), strides=(2,2), padding='same', activation='tanh') # 128x128
], name="Generator")


# 2. DISCRIMINATOR: Maps Image (128,128,3) -> Binary Real/Fake (1)
discriminator = Sequential([
    Conv2D(64, (4,4), strides=(2,2), padding='same', input_shape=(128, 128, 3)),
    LeakyReLU(alpha=0.2),
    Dropout(0.3),
    
    Conv2D(128, (4,4), strides=(2,2), padding='same'),
    LeakyReLU(alpha=0.2),
    Dropout(0.3),
    
    Conv2D(256, (4,4), strides=(2,2), padding='same'),
    LeakyReLU(alpha=0.2),
    Dropout(0.3),
    
    Flatten(),
    Dense(1, activation='sigmoid') # Is it real? 1. Is it fake? 0
], name="Discriminator")

discriminator.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
discriminator.trainable = False # We lock discriminator inside GAN structure loop

# 3. CONNECT THE GAN
gan = Sequential([generator, discriminator])
gan.compile(loss='binary_crossentropy', optimizer='adam')

print("Training GAN simultaneously for 500 steps... (This takes a moment)")
batch_size = 32
# GAN generators using tanh need images mapped [-1, 1] not [0, 1]
X_train_gan = (X_train * 2.0) - 1.0 

for step in range(500):
    # a. Train the Discriminator with half batch Real, half batch Fake
    idx = np.random.randint(0, X_train_gan.shape[0], batch_size)
    real_imgs = X_train_gan[idx]
    
    noise_batch = np.random.randn(batch_size, latent_dim)
    fake_imgs = generator.predict(noise_batch, verbose=0)
    
    # Label Smoothing (Real = 0.9, Fake = 0.0) speeds up GAN stability
    d_loss_real = discriminator.train_on_batch(real_imgs, np.ones((batch_size, 1)) * 0.9)
    d_loss_fake = discriminator.train_on_batch(fake_imgs, np.zeros((batch_size, 1)))
    
    # b. Train Generator via GAN structure (Aiming to trick locked discriminator to Output 1)
    noise_batch_g = np.random.randn(batch_size, latent_dim)
    g_loss = gan.train_on_batch(noise_batch_g, np.ones((batch_size, 1)))
    
    if (step+1) % 100 == 0:
        print(f"Step {step+1}/500 - Generator Loss: {g_loss:.4f} | Disc. Loss (Real {d_loss_real[0]:.4f}, Fake {d_loss_fake[0]:.4f})")

# 4. Synthesize 11 generic images at conclusion of 500 steps
viz_noise = np.random.randn(11, latent_dim)
gen_outputs = generator.predict(viz_noise)
gen_outputs = (gen_outputs + 1.0) / 2.0 # Standardize back out to [0, 1] space for display

plt.figure(figsize=(16, 3))
for i in range(11):
    plt.subplot(1, 11, i+1)
    plt.imshow(gen_outputs[i])
    plt.title(f"Gen {i+1}")
    plt.axis('off')
plt.suptitle("GAN Generated Content After Short Optimization Timeline")
plt.tight_layout()
plt.show()


# SECTION 9 — CNN Trained on GAN-Augmented Data
We command our customized Generative Adversarial Network to spit out 110 unclassified fake planetary artifacts. We inject these synthetic pictures back into the real dataset then retrain our CNN structural block from Section 5. 
*Note restricted scope for BTech task:* Since our beginner GAN does not synthesize matching *specific* classes (Known as Conditional GANs), we will just distribute them completely randomly as an educational experiment. 
*Syllabus Unit Covered: Generative Augmentation Implementation (Unit 5)*


In [ ]:
# 1. Create a larger payload of generated shapes
huge_noise = np.random.randn(110, latent_dim)
gan_110 = generator.predict(huge_noise)
gan_110 = (gan_110 + 1.0) / 2.0

# 2. Distribute randomly across the 11 classes as a baseline proof-of-concept
fake_labels_int = np.random.randint(0, 11, 110)
fake_labels_cat = tf.keras.utils.to_categorical(fake_labels_int, num_classes=11)

# 3. Compile Master Set
X_gan_augmented = np.vstack([X_train, gan_110])
y_gan_augmented = np.vstack([y_train, fake_labels_cat])

# 4. Test vanilla CNN behavior against new synthesized noise maps
gan_cnn = build_cnn()
print("Training CNN with GAN synthesized elements mixed into origin dataset ...")
history_gan_cnn = gan_cnn.fit(X_gan_augmented, y_gan_augmented, epochs=15, validation_data=(X_test, y_test), batch_size=32, verbose=1)

plot_history(history_gan_cnn, "CNN with GAN Content")
test_acc_gancnn = gan_cnn.evaluate(X_test, y_test, verbose=0)[1]
global_results['CNN (GAN + Real Data)'] = test_acc_gancnn

# 5. Summary Table comparison of standard CNN variants
import pandas as pd

temp_data = [
    ["CNN (Original Data)", test_acc_cnn],
    ["CNN (Augmented Data)", test_acc_aug],
    ["CNN (GAN + Real Data)", test_acc_gancnn],
    ["Transfer Learning (EfficientNetB0)", test_acc_tl]
]

print("\n--- ALL CNN DATASET COMPARISON TABLE ---")
df = pd.DataFrame(temp_data, columns=["Model", "Test Accuracy"])
print(df.to_string(index=False))


# SECTION 10 — CNN + LSTM Hybrid Classifier (Unit 4)
Long Short-Term Memory (LSTM) systems usually solve temporal inputs (words over time, stock markers over days). But if we flatten image features dynamically extracted by a standard spatial CNN block into a "sequence array", LSTMs scan those sequences tracing patterns from top-to-bottom patches across pixels, creating an immensely potent hybrid mechanism tracing spatial geometry.
*Syllabus Unit Covered: Recurrent Architectures and Hybrid Network Topologies (Unit 4)*


In [ ]:
from tensorflow.keras.layers import LSTM, Reshape

# 1. Harvest original frozen CNN layers up to its final convolution feature mapping pool 
hybrid_extractor = Sequential()
for layer in original_cnn.layers:
    # We strip out the dense end nodes, preserving only CNN block maps
    if isinstance(layer, tf.keras.layers.Flatten):
        break # Stomp before flattening
    hybrid_extractor.add(layer)

hybrid_extractor.trainable = False # Lock existing knowledge

# 2. Append sequence processing blocks correctly
hybrid_model = Sequential([
    hybrid_extractor,
    # Outputs dimension ~ (16, 16, 128)
    # The -1 flattens the two 16x16 spatial axes automatically providing 256 time steps holding 128 traits.
    Reshape((-1, 128)),
    
    LSTM(128, return_sequences=False), # Recurrently parses 'spatial sequence time' features
    Dense(64, activation='relu'),
    Dense(11, activation='softmax')
])

hybrid_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("Training CNN Spatial mapping sequence into an LSTM network...")
history_hybrid = hybrid_model.fit(X_train, y_train, epochs=20, validation_data=(X_test, y_test), batch_size=32, verbose=1)

plot_history(history_hybrid, "CNN+LSTM")
test_acc_hybrid = hybrid_model.evaluate(X_test, y_test, verbose=0)[1]
global_results['CNN + LSTM Hybrid'] = test_acc_hybrid
print(f"Hybrid Accuracy Output: {test_acc_hybrid:.4f}")


# SECTION 11 — Optimizer Comparison (Unit 3)
An optimizer governs how aggressively network error rates calculate Backpropagation slope changes.
* SGD (Stochastic Gradient Descent): Makes slow, extremely rigid single steps down the learning slope.
* RMSProp: Uses 'adaptive learning' reducing rate dynamically when slopes curve aggressively preventing blowouts.
* Adam: Incorporates both adaptive adjustments and geometric 'Momentum' pushing the gradient past tiny false potholes smoothly.
*Syllabus Unit Covered: Training Iteration and Slope Optimization Algorithms (Unit 3)*


In [ ]:
# List of optimization setups
opts = ['sgd', 'rmsprop', 'adam']
opt_metrics = {}

plt.figure(figsize=(10, 5))

# Iterate optimizers
for opt in opts:
    opt_model = build_cnn()
    opt_model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    
    print(f"Executing brief parameter check compiling purely under '{opt.upper()}' ...")
    # Reduced run to 15 epochs standard testing 
    h = opt_model.fit(X_train, y_train, epochs=15, validation_data=(X_test, y_test), batch_size=32, verbose=0)
    
    final_acc = h.history['val_accuracy'][-1]
    opt_metrics[opt.upper()] = final_acc
    plt.plot(h.history['val_accuracy'], label=opt.upper())

plt.title("Optimizer Validation Accuracy Racing Track", fontsize=14)
plt.ylabel("Validation Score Growth")
plt.xlabel("Epoch Increment Step")
plt.legend()
plt.show()

# Conclusion Optimizer Table
import pandas as pd
print("\n--- Optimizational Score Verification Table ---")
df_opts = pd.DataFrame(opt_metrics.items(), columns=["Optimizer Algorithm", "Score @ Epoch 15"])
print(df_opts.to_string(index=False))


# SECTION 12 — Final Results and Conclusion
All models have been evaluated on their test sets. This unified result table provides an immediate look across varying algorithmic topologies ranging from standard connected MLPs to deep Convolutional feature extractors feeding Recurrent LSTM models.

### Conclusion

1. **Which model performed best and why:** `Transfer Learning (EfficientNetB0)` is typically the strongest architecture here. While custom CNNs were guessing edge shapes blindly from only ~1300 small images, the transfer network reused million-parameter pre-calculated feature recognizers developed over time by big tech labs allowing high validation efficiency easily. 
2. **What GAN contributed:** The Generative system provided an automated capability of manufacturing brand new instances into the dataset matrix preventing data scarcity starvation.
3. **Limitations of the project:** GAN systems demand massive overhead running 10,000+ iteration cycles over 24+ hours creating high-fidelity results. Our Colab limit capping at 500 Steps ensures purely simplified blur generation. Finally, lack of "Conditional targeting" inside our demo generator structure limits targeted class augmentation. 
4. **Future scope:** The future application implies utilizing massive telescope array datasets running targeted Conditional GANs outputting completely verified class instances for data manipulation. Also, appending Multi-Head Attention Mechanisms inside models isolating planetary phenomena (Saturn’s specific atmospheric bands against rings).


In [ ]:
# Master Scoreboard Print Sequence
import pandas as pd

# Iterate and structure format
master_data = []
for key, val in global_results.items():
    master_data.append([key, f"{val*100:.2f}%"])

df_master = pd.DataFrame(master_data, columns=["Model Topology", "Test Map Output Score"])

print("="*60)
print("              MASTER ANALYSIS ARCHITECTURE SCOREBOARD")
print("="*60)
print(df_master.to_string(index=False))
print("="*60)
